# NB21 · OOP Advanced

`advanced/` · notebook 21 of 32 · 42 questions + a mini-project

In NB14 you wrote classes with `__init__`, methods, `__str__`, `__repr__` and
inheritance. This notebook shows how to make your own objects behave like
Python's built-in ones: they print nicely, compare and sort, add with `+`,
work with `len()`, `in` and `[]`, and can be called like functions.

You'll also meet the tools seniors reach for to cut boilerplate:
properties, `@classmethod`, `@dataclass`, `enum`, abstract base classes and
mixins. The traps here (a class that loses its hash, a property that calls
itself forever, a shared mutable default) are ones that show up in real
code reviews.

**One limit:** `iter()`, `next()` and `yield` arrive in NB22. When a class
here needs `__iter__`, the solution returns a generator expression (NB10).

## Prerequisites

Before you start, you should be comfortable with:

- [ ] NB14: OOP basics
- [ ] NB20: intermediate capstone

## What you'll practise

- __repr__, __str__, __format__
- The __eq__ and __hash__ contract; ordering with total_ordering
- Operator overloading: __add__, __radd__, __iadd__, returning NotImplemented
- Container protocol: __len__, __getitem__, __setitem__, __contains__, __iter__; __call__, __bool__
- Properties: getter, setter, validation, computed values
- classmethod alternative constructors; staticmethod
- dataclasses: field(default_factory), frozen, order, __post_init__, slots, kw_only
- enum: Enum, IntEnum, StrEnum, Flag, auto
- Abstract base classes; multiple inheritance, MRO, cooperative super(); mixins; __slots__
- Subclassing built-ins (and why UserDict exists); copy.replace() (3.13)

## How to use this notebook

1. Read the question and write your code in the cell beneath it.
2. Run the cell, then open **Expected output** and compare.
3. Stuck? Open **Hint** first. Open **Solution** only after a real attempt.
4. Once it works, open **Senior dev solution** to see how an experienced developer would write it, and why.

The **Solution** only uses what this notebook and the ones before it teach. The **Senior dev solution** sometimes uses later topics on purpose, and says which notebook covers them.

| Type | What you do |
|---|---|
| **Write** | Write code that prints the answer |
| **Predict** | Read the code and predict exactly what it prints |
| **Error** | Predict which error the code raises |
| **Fix** | The code has a bug. Find it and fix it |
| **Refactor** | The code works. Make it cleaner without changing its output |

| Level | Meaning |
|---|---|
| L1 | Extremely Easy: one idea, a line or two |
| L2 | Easy: one idea with a small twist |
| L3 | Intermediate: two or three ideas combined |
| L4 | Somewhat Difficult: needs care to get exactly right |
| L5 | Hard: several ideas and a trap or two |

---

## L1 · Extremely Easy

### NB21-Q01 · L1 · __repr__ and __str__ · Write

Write a `Point` class that stores `x` and `y`, with a `__repr__` that
returns text like `Point(x=3, y=4)`. Create `Point(3, 4)` and print it.

<details><summary><b>Hint</b></summary>

`__repr__` must **return** a string. When a class has no `__str__`, `print()` falls back to `__repr__`.

</details>

<details><summary><b>Expected output</b></summary>

```text
Point(x=3, y=4)
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point(x={self.x}, y={self.y})"


p = Point(3, 4)
print(p)
```

`print()` asks the object for `str(p)`. With no `__str__`, Python uses `__repr__` instead, so one method covers both. A good `__repr__` looks like the code that would rebuild the object.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"{type(self).__name__}(x={self.x!r}, y={self.y!r})"


p = Point(3, 4)
print(p)
```

Two habits. `!r` uses each value's own repr, so a string shows its quotes (`Point(x='3', ...)`) and you can see the type bug at a glance. `type(self).__name__` means a subclass prints its own name instead of lying and saying `Point`.

</details>

In [ ]:
# your code here


### NB21-Q02 · L1 · __repr__ and __str__ · Predict

A class with both `__str__` and `__repr__`. Predict the output.

**What does this print?**

```python
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price

    def __str__(self):
        return f"{self.name} (${self.price:.2f})"

    def __repr__(self):
        return f"Product({self.name!r}, {self.price!r})"


tea = Product("Tea", 3.5)
print(tea)
print(repr(tea))
print([tea])
print(f"{tea} / {tea!r}")
```

<details><summary><b>Hint</b></summary>

`print()` and f-strings use `__str__`. `repr()` and `!r` use `__repr__`. What does a list use for the items inside it?

</details>

<details><summary><b>Expected output</b></summary>

```text
Tea ($3.50)
Product('Tea', 3.5)
[Product('Tea', 3.5)]
Tea ($3.50) / Product('Tea', 3.5)
```

</details>

<details><summary><b>Solution</b></summary>

`print(tea)` uses `__str__`: `Tea ($3.50)`. `repr(tea)` gives `Product('Tea', 3.5)`. A list always shows its items with `__repr__`, so `[tea]` prints the repr form. In the f-string, `{tea}` uses `__str__` and `{tea!r}` uses `__repr__`.

</details>

<details><summary><b>Senior dev solution</b></summary>

`__str__` is for users (receipts, messages). `__repr__` is for developers (logs, debuggers, lists). If you only write one, write `__repr__`: it's the one you see when something goes wrong.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB21-Q03 · L1 · Properties · Write

Write a `Rectangle` class with `width` and `height`, and a read-only
`area` **property** (no brackets when you use it). Print the area of a
`4` by `2.5` rectangle.

<details><summary><b>Hint</b></summary>

Put `@property` above a method called `area`. Then `r.area` runs the method.

</details>

<details><summary><b>Expected output</b></summary>

```text
10.0
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self):
        return self.width * self.height


r = Rectangle(4, 2.5)
print(r.area)
```

`@property` turns a method into something you read like an attribute. The area is worked out each time you ask, so it's always right even after `width` changes. Without a setter, `r.area = 5` raises `AttributeError`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self):
        """Area in square units, always in step with width and height."""
        return self.width * self.height


r = Rectangle(4, 2.5)
print(r.area)
```

Seniors use a property for values that are *derived* from other attributes. Storing `self.area` in `__init__` would go stale the moment someone changes `width`. A docstring on the property shows up in `help(Rectangle)`.

</details>

In [ ]:
# your code here


### NB21-Q04 · L1 · enum · Write

Make an `Enum` called `Size` with members `SMALL`, `MEDIUM` and `LARGE`,
using `auto()` for the values. Print the `MEDIUM` member, its name and its
value on one line.

<details><summary><b>Hint</b></summary>

`from enum import Enum, auto`. Each member has `.name` and `.value`.

</details>

<details><summary><b>Expected output</b></summary>

```text
Size.MEDIUM MEDIUM 2
```

</details>

<details><summary><b>Solution</b></summary>

```python
from enum import Enum, auto


class Size(Enum):
    SMALL = auto()
    MEDIUM = auto()
    LARGE = auto()


print(Size.MEDIUM, Size.MEDIUM.name, Size.MEDIUM.value)
```

An `Enum` is a fixed set of named constants. `auto()` numbers them from `1`, so `MEDIUM` is `2`. Printing a member shows `Size.MEDIUM`, its `.name` is the text `MEDIUM`, and `.value` is `2`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from enum import Enum, auto


class Size(Enum):
    SMALL = auto()
    MEDIUM = auto()
    LARGE = auto()


size = Size["MEDIUM"]
print(size, size.name, size.value)
```

`Size["MEDIUM"]` looks a member up by name, which is how you turn text from a config file into a member. Seniors compare members (`size is Size.MEDIUM`), never their values: with `auto()` the numbers are an implementation detail.

</details>

In [ ]:
# your code here


---

## L2 · Easy

### NB21-Q05 · L2 · Operator overloading · Write

Write a `Vector` class with `x` and `y`. Give it `__add__` (vector + vector),
`__mul__` (vector * number) and a `__repr__`. Then print:

```text
Vector(4, 6)
Vector(3, 6)
```

for `Vector(1, 2) + Vector(3, 4)` and `Vector(1, 2) * 3`.

<details><summary><b>Hint</b></summary>

`a + b` calls `a.__add__(b)`. Each method should **return a new** `Vector`, not change `self`.

</details>

<details><summary><b>Expected output</b></summary>

```text
Vector(4, 6)
Vector(3, 6)
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

    def __mul__(self, number):
        return Vector(self.x * number, self.y * number)

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"


print(Vector(1, 2) + Vector(3, 4))
print(Vector(1, 2) * 3)
```

Python turns `a + b` into `a.__add__(b)` and `a * 3` into `a.__mul__(3)`. Returning a new object keeps vectors behaving like numbers: `a + b` never changes `a`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return Vector(self.x + other.x, self.y + other.y)

    def __mul__(self, number):
        if not isinstance(number, (int, float)):
            return NotImplemented
        return Vector(self.x * number, self.y * number)

    __rmul__ = __mul__

    def __repr__(self):
        return f"Vector({self.x!r}, {self.y!r})"


print(Vector(1, 2) + Vector(3, 4))
print(Vector(1, 2) * 3)
```

Returning `NotImplemented` for the wrong type lets Python give a clean `TypeError` (see Q11) instead of a confusing `AttributeError`. `__rmul__ = __mul__` makes `3 * v` work too, since `int` doesn't know about vectors.

</details>

In [ ]:
# your code here


### NB21-Q06 · L2 · classmethod and staticmethod · Refactor

`rgb_from_hex` builds an `Rgb` from text like `"#ff8800"`, but it lives
outside the class. Turn it into an alternative constructor:
`Rgb.from_hex(text)`. Keep the output the same.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
class Rgb:
    def __init__(self, red, green, blue):
        self.red = red
        self.green = green
        self.blue = blue

    def __repr__(self):
        return f"Rgb({self.red}, {self.green}, {self.blue})"


def rgb_from_hex(text):
    digits = text.lstrip("#")
    return Rgb(int(digits[0:2], 16), int(digits[2:4], 16), int(digits[4:6], 16))


print(rgb_from_hex("#ff8800"))
print(rgb_from_hex("#1e90ff"))
```

<details><summary><b>Hint</b></summary>

A `@classmethod` receives the class itself as its first argument, usually called `cls`. `cls(...)` builds a new object.

</details>

<details><summary><b>Expected output</b></summary>

```text
Rgb(255, 136, 0)
Rgb(30, 144, 255)
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Rgb:
    def __init__(self, red, green, blue):
        self.red = red
        self.green = green
        self.blue = blue

    @classmethod
    def from_hex(cls, text):
        digits = text.lstrip("#")
        return cls(int(digits[0:2], 16), int(digits[2:4], 16), int(digits[4:6], 16))

    def __repr__(self):
        return f"Rgb({self.red}, {self.green}, {self.blue})"


print(Rgb.from_hex("#ff8800"))
print(Rgb.from_hex("#1e90ff"))
```

A class can only have one `__init__`, so other ways of building an object become classmethods named `from_...`. Using `cls(...)` instead of `Rgb(...)` means a subclass calling `from_hex` gets a subclass object.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Rgb:
    def __init__(self, red, green, blue):
        self.red = red
        self.green = green
        self.blue = blue

    @classmethod
    def from_hex(cls, text):
        digits = text.removeprefix("#")
        return cls(*(int(digits[i:i + 2], 16) for i in (0, 2, 4)))

    def __repr__(self):
        return f"Rgb({self.red}, {self.green}, {self.blue})"


print(Rgb.from_hex("#ff8800"))
print(Rgb.from_hex("#1e90ff"))
```

`removeprefix("#")` removes exactly one `#`, while `lstrip("#")` removes every `#` at the start, which hides bad input. The standard library is full of this pattern: `dict.fromkeys`, `datetime.fromisoformat`, `int.from_bytes`.

</details>

In [ ]:
class Rgb:
    def __init__(self, red, green, blue):
        self.red = red
        self.green = green
        self.blue = blue

    def __repr__(self):
        return f"Rgb({self.red}, {self.green}, {self.blue})"


def rgb_from_hex(text):
    digits = text.lstrip("#")
    return Rgb(int(digits[0:2], 16), int(digits[2:4], 16), int(digits[4:6], 16))


print(rgb_from_hex("#ff8800"))
print(rgb_from_hex("#1e90ff"))


### NB21-Q07 · L2 · classmethod and staticmethod · Write

Give a `Password` class a **static method** `is_strong(text)` that returns
`True` when the text is at least 8 characters long and contains a digit.
Call it on the class (no object needed) for `"sunshine"` and `"sunsh1ne!"`
and print both results on one line.

<details><summary><b>Hint</b></summary>

Put `@staticmethod` above the method. A static method gets no `self` and no `cls`: only the arguments you pass.

</details>

<details><summary><b>Expected output</b></summary>

```text
False True
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Password:
    @staticmethod
    def is_strong(text):
        has_digit = any(ch.isdigit() for ch in text)
        return len(text) >= 8 and has_digit


print(Password.is_strong("sunshine"), Password.is_strong("sunsh1ne!"))
```

A static method is a plain function that lives inside the class because it belongs with it. It doesn't need an object, so you call it on the class. `"sunshine"` is long enough but has no digit.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
MIN_LENGTH = 8


class Password:
    @staticmethod
    def is_strong(text):
        return len(text) >= MIN_LENGTH and any(ch.isdigit() for ch in text)


print(Password.is_strong("sunshine"), Password.is_strong("sunsh1ne!"))
```

Seniors use `@staticmethod` sparingly. If the function doesn't need the class at all, a module-level function is often clearer. It earns its place when callers naturally look for it on the class, as here.

</details>

In [ ]:
# your code here


### NB21-Q08 · L2 · __eq__ and __hash__ · Predict

One class compares by identity, the other by value. Predict the output.

**What does this print?**

```python
class Plain:
    def __init__(self, sku):
        self.sku = sku


class Compared:
    def __init__(self, sku):
        self.sku = sku

    def __eq__(self, other):
        return isinstance(other, Compared) and self.sku == other.sku


print(Plain("A1") == Plain("A1"))
print(Compared("A1") == Compared("A1"))
print(Compared("A1") is Compared("A1"))
print(Compared("A1") != Compared("B2"))
```

<details><summary><b>Hint</b></summary>

Without `__eq__`, `==` falls back to `is`. And `is` never looks at `__eq__`.

</details>

<details><summary><b>Expected output</b></summary>

```text
False
True
False
True
```

</details>

<details><summary><b>Solution</b></summary>

`Plain` has no `__eq__`, so `==` checks identity: two separate objects, `False`. `Compared` compares `sku`, so `True`. `is` still checks identity, so `False`. Python builds `!=` from your `__eq__` automatically, so `"A1" != "B2"` gives `True`.

</details>

<details><summary><b>Senior dev solution</b></summary>

When you define `__eq__`, you also decide whether the object can be hashed (see Q14 and Q15). That's why seniors usually let `@dataclass` write `__eq__` for them.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB21-Q09 · L2 · dataclasses · Write

Write a dataclass `InventoryItem` with `name` (str), `price` (float) and
`qty` (int, default `0`), plus a method `total()` returning
`price * qty`. Print `InventoryItem("pen", 1.5, 4)`, then its total,
then `InventoryItem("pad", 2.0)`.

<details><summary><b>Hint</b></summary>

`from dataclasses import dataclass`. Fields are written as `name: type` lines in the class body; a default goes after `=`.

</details>

<details><summary><b>Expected output</b></summary>

```text
InventoryItem(name='pen', price=1.5, qty=4)
6.0
InventoryItem(name='pad', price=2.0, qty=0)
```

</details>

<details><summary><b>Solution</b></summary>

```python
from dataclasses import dataclass


@dataclass
class InventoryItem:
    name: str
    price: float
    qty: int = 0

    def total(self):
        return self.price * self.qty


pen = InventoryItem("pen", 1.5, 4)
print(pen)
print(pen.total())
print(InventoryItem("pad", 2.0))
```

`@dataclass` reads the annotated fields and writes `__init__`, `__repr__` and `__eq__` for you. Fields with defaults must come after fields without them, exactly like function parameters.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from dataclasses import dataclass


@dataclass(slots=True)
class InventoryItem:
    name: str
    price: float
    qty: int = 0

    def total(self):
        return self.price * self.qty


pen = InventoryItem("pen", 1.5, 4)
print(pen)
print(pen.total())
print(InventoryItem("pad", 2.0))
```

`slots=True` uses `__slots__` (Q27): less memory per object, and a typo like `pen.qyt = 5` raises an error instead of quietly creating a new attribute. The type annotations aren't checked at runtime; tools like mypy check them (NB26).

</details>

In [ ]:
# your code here


### NB21-Q10 · L2 · Container protocol · Write

Write a `Playlist` class that stores a list of songs and has a `__len__`
method. Print `len()` and `bool()` for an empty playlist and for one with
two songs:

```text
0 False
2 True
```

Don't write a `__bool__` method.

<details><summary><b>Hint</b></summary>

`len(obj)` calls `obj.__len__()`. When a class has no `__bool__`, Python decides truthiness from `__len__`.

</details>

<details><summary><b>Expected output</b></summary>

```text
0 False
2 True
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Playlist:
    def __init__(self, songs):
        self.songs = list(songs)

    def __len__(self):
        return len(self.songs)


empty = Playlist([])
road_trip = Playlist(["Holiday", "Africa"])
print(len(empty), bool(empty))
print(len(road_trip), bool(road_trip))
```

`bool(x)` tries `__bool__` first. When that's missing it uses `__len__`: length `0` is falsy, anything else is truthy. So `if playlist:` already works the way you'd expect for a container.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Playlist:
    def __init__(self, songs):
        self.songs = list(songs)

    def __len__(self):
        return len(self.songs)


empty = Playlist([])
road_trip = Playlist(["Holiday", "Africa"])
print(len(empty), bool(empty))
print(len(road_trip), bool(road_trip))
```

Same code. Copying with `list(songs)` matters: it stops the playlist from sharing the caller's list. Seniors only add `__bool__` when truthiness should mean something other than "not empty".

</details>

In [ ]:
# your code here


### NB21-Q11 · L2 · Operator overloading · Error

`__add__` returns `NotImplemented` for anything that isn't `Money`. Which error does the last line raise?

**Which error does this raise?**

```python
class Money:
    def __init__(self, cents):
        self.cents = cents

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return Money(self.cents + other.cents)


total = Money(500) + Money(250)
print(total.cents)
print(Money(500) + 3)
```

<details><summary><b>Hint</b></summary>

`NotImplemented` is a special *value*, not an exception. It tells Python to try the other side's method. What happens when that fails too?

</details>

<details><summary><b>Expected output</b></summary>

```text
750
TypeError
```

</details>

<details><summary><b>Solution</b></summary>

`TypeError`. `Money.__add__` returns `NotImplemented`, so Python tries `int.__radd__(3, money)`, which also gives up. With both sides out of ideas, Python raises `TypeError: unsupported operand type(s)`. Don't confuse this with `NotImplementedError`, an exception you raise for a method a subclass must provide.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Money:
    def __init__(self, cents):
        self.cents = cents

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return Money(self.cents + other.cents)


print((Money(500) + Money(3)).cents)
```

Returning `NotImplemented` (instead of raising) is polite: it gives the *other* object a chance to handle the operation, which is how `Decimal + int` and `3 * vector` work.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB21-Q12 · L2 · enum · Predict

`Enum`, `IntEnum` and `StrEnum` behave differently. Predict the output.

**What does this print?**

```python
from enum import Enum, IntEnum, StrEnum, auto


class Color(Enum):
    RED = 1
    GREEN = 2


class Priority(IntEnum):
    LOW = 1
    HIGH = 2


class Status(StrEnum):
    OPEN = auto()
    CLOSED = auto()


print(Color.RED == 1, Priority.LOW == 1)
print(Priority.HIGH > Priority.LOW, Priority.HIGH + 10)
print(Status.OPEN == "open", Status.CLOSED.upper())
print(Color(2) is Color.GREEN)
```

<details><summary><b>Hint</b></summary>

A plain `Enum` member is not a number. `IntEnum` members *are* ints, and `StrEnum` members *are* strings. With `StrEnum`, `auto()` uses the lowercase name.

</details>

<details><summary><b>Expected output</b></summary>

```text
False True
True 12
True CLOSED
True
```

</details>

<details><summary><b>Solution</b></summary>

`Color.RED == 1` is `False`: a plain enum member only equals itself. `Priority.LOW == 1` is `True` because `IntEnum` members are ints, so they compare and do maths like ints (`HIGH + 10` is `12`). `StrEnum` members are strings and `auto()` gives `"open"` and `"closed"`. `Color(2)` looks a member up by value.

</details>

<details><summary><b>Senior dev solution</b></summary>

Seniors prefer plain `Enum` because it stops accidental comparisons with raw numbers. `IntEnum` and `StrEnum` are for when the value must go somewhere that expects an int or str: HTTP codes, JSON, database columns.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB21-Q13 · L2 · enum · Write

Make a `Flag` enum `Permission` with `READ`, `WRITE` and `EXECUTE` (use
`auto()`). A user starts with read and write. Print:

```text
True False
READ|WRITE|EXECUTE
READ|EXECUTE 5
```

Line 1: does the user have `WRITE`? `EXECUTE`? Line 2: the `.name` after
adding `EXECUTE`. Line 3: the `.name` and `.value` after removing `WRITE`.

<details><summary><b>Hint</b></summary>

Combine flags with `|`, test with `in`, and remove one with `&= ~Flag.MEMBER` (the bitwise operators from NB16).

</details>

<details><summary><b>Expected output</b></summary>

```text
True False
READ|WRITE|EXECUTE
READ|EXECUTE 5
```

</details>

<details><summary><b>Solution</b></summary>

```python
from enum import Flag, auto


class Permission(Flag):
    READ = auto()
    WRITE = auto()
    EXECUTE = auto()


user = Permission.READ | Permission.WRITE
print(Permission.WRITE in user, Permission.EXECUTE in user)
user |= Permission.EXECUTE
print(user.name)
user &= ~Permission.WRITE
print(user.name, user.value)
```

With `Flag`, `auto()` gives powers of two (`1`, `2`, `4`), so members combine like bits. `READ | EXECUTE` is `1 + 4 = 5`. `in` tests whether a bit is set, and `& ~WRITE` clears one bit.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from enum import Flag, auto


class Permission(Flag):
    READ = auto()
    WRITE = auto()
    EXECUTE = auto()
    READ_WRITE = READ | WRITE


user = Permission.READ_WRITE
print(Permission.WRITE in user, Permission.EXECUTE in user)
user |= Permission.EXECUTE
print(user.name)
user &= ~Permission.WRITE
print(user.name, user.value)
```

Named combinations like `READ_WRITE` document common sets of permissions. Compared with raw bitmask ints (NB16), a `Flag` prints readable names in logs, which makes permission bugs far easier to spot.

</details>

In [ ]:
# your code here


---

## L3 · Intermediate

### NB21-Q14 · L3 · __eq__ and __hash__ · Error

`Point` defines `__eq__` and nothing else. Which error does this raise?

**Which error does this raise?**

```python
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        return (self.x, self.y) == (other.x, other.y)


print(Point(1, 2) == Point(1, 2))
visited = {Point(1, 2), Point(1, 2)}
print(len(visited))
```

<details><summary><b>Hint</b></summary>

Sets and dict keys need `hash()`. What happens to the inherited `__hash__` when a class defines `__eq__`?

</details>

<details><summary><b>Expected output</b></summary>

```text
True
TypeError
```

</details>

<details><summary><b>Solution</b></summary>

`TypeError: unhashable type: 'Point'`. Equal objects must have equal hashes. The default hash is based on identity, which would break that rule, so defining `__eq__` sets `__hash__` to `None`. The comparison on the first line still works.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        return (self.x, self.y) == (other.x, other.y)


print(Point.__hash__ is None)
```

`Point.__hash__ is None` shows what happened. Seniors make this decision on purpose: mutable objects stay unhashable, immutable ones get a `__hash__` (Q15) or use `@dataclass(frozen=True)`.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB21-Q15 · L3 · __eq__ and __hash__ · Write

Make `Point` (with `x` and `y`) compare **and** hash by value. Put these
five points in a set, then print how many distinct points there are and
whether `Point(1, 2)` is in the set: `3 True`.

**Starting code (already in the cell below):**

```python
coords = [(1, 2), (2, 1), (1, 2), (0, 0), (2, 1)]
```

<details><summary><b>Hint</b></summary>

`__hash__` should hash the same values `__eq__` compares. Hashing a tuple `(self.x, self.y)` does that.

</details>

<details><summary><b>Expected output</b></summary>

```text
3 True
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))


seen = {Point(x, y) for x, y in coords}
print(len(seen), Point(1, 2) in seen)
```

A set first uses the hash to find a bucket, then `__eq__` to confirm a match. Hashing the same tuple that `__eq__` compares guarantees equal points land in the same bucket. Two of the five points are duplicates, leaving `3`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from dataclasses import dataclass


@dataclass(frozen=True)
class Point:
    x: int
    y: int


seen = {Point(x, y) for x, y in coords}
print(len(seen), Point(1, 2) in seen)
```

`frozen=True` writes `__eq__` and `__hash__` together and blocks changes to `x` and `y`. That matters: if a hashed object changes, it sits in the wrong bucket and the set can no longer find it.

</details>

In [ ]:
coords = [(1, 2), (2, 1), (1, 2), (0, 0), (2, 1)]

# your code here


### NB21-Q16 · L3 · Properties · Write

Write a `Temperature` class with a `celsius` property whose setter raises
`ValueError("below absolute zero")` for values under `-273.15`, and a
read-only `fahrenheit` property. Run this script with your class:

```py
t = Temperature(20)
print(t.fahrenheit)
t.celsius = 100
print(t.fahrenheit)
try:
    t.celsius = -300
except ValueError as e:
    print("rejected:", e)
print(t.celsius)
```

<details><summary><b>Hint</b></summary>

Store the real value in `self._celsius`. In `__init__`, assign through `self.celsius = ...` so the setter checks the starting value too.

</details>

<details><summary><b>Expected output</b></summary>

```text
68.0
212.0
rejected: below absolute zero
100
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("below absolute zero")
        self._celsius = value

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32


t = Temperature(20)
print(t.fahrenheit)
t.celsius = 100
print(t.fahrenheit)
try:
    t.celsius = -300
except ValueError as e:
    print("rejected:", e)
print(t.celsius)
```

`@celsius.setter` runs whenever someone assigns `t.celsius = ...`, so every change is checked, including the one in `__init__`. The bad value is rejected before it's stored, so `t.celsius` is still `100`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
ABSOLUTE_ZERO_C = -273.15


class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < ABSOLUTE_ZERO_C:
            raise ValueError("below absolute zero")
        self._celsius = value

    @property
    def fahrenheit(self):
        return self.celsius * 9 / 5 + 32


t = Temperature(20)
print(t.fahrenheit)
t.celsius = 100
print(t.fahrenheit)
try:
    t.celsius = -300
except ValueError as e:
    print("rejected:", e)
print(t.celsius)
```

Properties let you start with a plain attribute and add validation later without changing any calling code. That's why Python code doesn't need Java-style `get_x()`/`set_x()` methods (see Q36).

</details>

In [ ]:
# your code here


### NB21-Q17 · L3 · Properties · Fix

This should print `100`, but it crashes. Fix it.

**Buggy code (copied into the cell below):**

```python
class Account:
    def __init__(self, balance):
        self.balance = balance

    @property
    def balance(self):
        return self.balance

    @balance.setter
    def balance(self, value):
        if value < 0:
            raise ValueError("negative balance")
        self.balance = value


acct = Account(100)
print(acct.balance)
```

<details><summary><b>Hint</b></summary>

Inside the setter, `self.balance = value` is itself an assignment to the property. What does that call?

</details>

<details><summary><b>Expected output</b></summary>

```text
100
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Account:
    def __init__(self, balance):
        self.balance = balance

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, value):
        if value < 0:
            raise ValueError("negative balance")
        self._balance = value


acct = Account(100)
print(acct.balance)
```

The original raises `RecursionError`. `self.balance = value` inside the setter calls the setter again, which assigns again, forever. The getter has the same problem. The property needs a *different* name to store the data: `self._balance`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Account:
    def __init__(self, balance):
        self.balance = balance

    @property
    def balance(self):
        """Current balance. Can never go negative."""
        return self._balance

    @balance.setter
    def balance(self, value):
        if value < 0:
            raise ValueError("negative balance")
        self._balance = value


acct = Account(100)
print(acct.balance)
```

The convention is fixed: the property is `balance`, the storage is `_balance`. The leading underscore (NB14) tells other developers "go through the property". Only the property's own methods touch `_balance`.

</details>

In [ ]:
class Account:
    def __init__(self, balance):
        self.balance = balance

    @property
    def balance(self):
        return self.balance

    @balance.setter
    def balance(self, value):
        if value < 0:
            raise ValueError("negative balance")
        self.balance = value


acct = Account(100)
print(acct.balance)


### NB21-Q18 · L3 · __format__ · Write

Write a `Distance` class (stores `meters`) with a `__format__` method that
understands these format specs:

- `"km"`: kilometres with 2 decimals, like `1.50 km`
- `"mi"`: miles with 2 decimals (1 mile = `1609.344` m), like `0.93 mi`
- `""` (nothing): the metres, like `1500 m`

For `Distance(1500)`, print `f"{d:km}"`, `f"{d:mi}"` and `f"{d}"` on one
line, separated by ` | `.

<details><summary><b>Hint</b></summary>

In `f"{d:km}"`, everything after the colon is passed to `d.__format__("km")` as a string. Return a string from each branch.

</details>

<details><summary><b>Expected output</b></summary>

```text
1.50 km | 0.93 mi | 1500 m
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Distance:
    def __init__(self, meters):
        self.meters = meters

    def __format__(self, spec):
        if spec == "km":
            return f"{self.meters / 1000:.2f} km"
        if spec == "mi":
            return f"{self.meters / 1609.344:.2f} mi"
        return f"{self.meters} m"


d = Distance(1500)
print(f"{d:km} | {d:mi} | {d}")
```

f-strings and `format()` call `__format__` with the text after the colon. An empty spec (`{d}`) passes `""`. Your method decides what each spec means, which is how `datetime` understands `{when:%Y-%m-%d}`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
METERS_PER = {"km": 1000, "mi": 1609.344}


class Distance:
    def __init__(self, meters):
        self.meters = meters

    def __format__(self, spec):
        if not spec:
            return f"{self.meters} m"
        if spec not in METERS_PER:
            raise ValueError(f"unknown unit {spec!r}")
        return f"{self.meters / METERS_PER[spec]:.2f} {spec}"


d = Distance(1500)
print(f"{d:km} | {d:mi} | {d}")
```

A lookup table makes adding units a one-line change, and an unknown spec raises `ValueError` like the built-in types do (`f"{5:km}"` fails too) instead of silently falling through to metres.

</details>

In [ ]:
# your code here


### NB21-Q19 · L3 · Ordering · Write

Software versions don't sort as text: `"1.10.0"` comes before `"1.9.3"`
alphabetically. Write a `Version` class built from text, with `__eq__`
and `__lt__`, and use `@total_ordering` to get the other comparisons.
`repr` should look like `Version('1.10.0')`. Print the sorted versions,
then whether `Version("2.0.0") >= Version("1.10.0")`.

**Starting code (already in the cell below):**

```python
releases = ["1.10.0", "1.2.0", "1.9.3"]
```

<details><summary><b>Hint</b></summary>

Store the version as a tuple of ints, e.g. `(1, 10, 0)`. Tuples compare item by item. `from functools import total_ordering`.

</details>

<details><summary><b>Expected output</b></summary>

```text
[Version('1.2.0'), Version('1.9.3'), Version('1.10.0')]
True
```

</details>

<details><summary><b>Solution</b></summary>

```python
from functools import total_ordering


@total_ordering
class Version:
    def __init__(self, text):
        self.text = text
        self.parts = tuple(int(part) for part in text.split("."))

    def __eq__(self, other):
        return self.parts == other.parts

    def __lt__(self, other):
        return self.parts < other.parts

    def __repr__(self):
        return f"Version({self.text!r})"


print(sorted(Version(r) for r in releases))
print(Version("2.0.0") >= Version("1.10.0"))
```

`(1, 9, 3) < (1, 10, 0)` because `9 < 10` as numbers. `sorted()` only needs `<`, but `>=` needs more: `@total_ordering` builds `<=`, `>` and `>=` from your `__eq__` and `__lt__`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from dataclasses import dataclass, field


@dataclass(frozen=True, order=True)
class Version:
    parts: tuple
    text: str = field(compare=False)

    @classmethod
    def parse(cls, text):
        return cls(tuple(int(part) for part in text.split(".")), text)

    def __repr__(self):
        return f"Version({self.text!r})"


print(sorted(Version.parse(r) for r in releases))
print(Version.parse("2.0.0") >= Version.parse("1.10.0"))
```

`@dataclass(order=True)` writes all six comparisons from the fields, and `field(compare=False)` leaves the original text out of them. Frozen also makes versions hashable, so they work as dict keys.

</details>

In [ ]:
releases = ["1.10.0", "1.2.0", "1.9.3"]

# your code here


### NB21-Q20 · L3 · Container protocol · Write

Write an `Inventory` class that wraps a dict of item counts:

- `inv[name]` gives the count, or `0` for an unknown item
- `inv[name] = count` stores it, but raises `ValueError` if the count is negative
- `name in inv` is `True` only when the count is above `0`
- `len(inv)` counts the items in stock (count above `0`)
- looping over `inv` gives the in-stock names in alphabetical order

Run this script:

```py
inv = Inventory()
inv["apple"] = 5
inv["pear"] = 0
inv["fig"] = 2
print(inv["apple"], inv["kiwi"])
print("pear" in inv, "fig" in inv)
print(len(inv), list(inv))
```

<details><summary><b>Hint</b></summary>

The methods are `__getitem__`, `__setitem__`, `__contains__`, `__len__` and `__iter__`. `__iter__` must return something you can loop over: a generator expression works.

</details>

<details><summary><b>Expected output</b></summary>

```text
5 0
False True
2 ['apple', 'fig']
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Inventory:
    def __init__(self):
        self._stock = {}

    def __getitem__(self, name):
        return self._stock.get(name, 0)

    def __setitem__(self, name, count):
        if count < 0:
            raise ValueError("count cannot be negative")
        self._stock[name] = count

    def __contains__(self, name):
        return self[name] > 0

    def __len__(self):
        return len([name for name in self._stock if self._stock[name] > 0])

    def __iter__(self):
        return (name for name in sorted(self._stock) if self._stock[name] > 0)


inv = Inventory()
inv["apple"] = 5
inv["pear"] = 0
inv["fig"] = 2
print(inv["apple"], inv["kiwi"])
print("pear" in inv, "fig" in inv)
print(len(inv), list(inv))
```

Each piece of syntax maps to one method: `inv[k]` to `__getitem__`, `inv[k] = v` to `__setitem__`, `in` to `__contains__`, `len()` to `__len__`, and `for`/`list()` to `__iter__`. `__contains__` reuses `self[name]`, so the "unknown means 0" rule lives in one place.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Inventory:
    def __init__(self):
        self._stock = {}

    def __getitem__(self, name):
        return self._stock.get(name, 0)

    def __setitem__(self, name, count):
        if count < 0:
            raise ValueError("count cannot be negative")
        self._stock[name] = count

    def __contains__(self, name):
        return self[name] > 0

    def _in_stock(self):
        return sorted(name for name, count in self._stock.items() if count > 0)

    def __len__(self):
        return len(self._in_stock())

    def __iter__(self):
        return iter(self._in_stock())


inv = Inventory()
inv["apple"] = 5
inv["pear"] = 0
inv["fig"] = 2
print(inv["apple"], inv["kiwi"])
print("pear" in inv, "fig" in inv)
print(len(inv), list(inv))
```

One private helper, `_in_stock()`, defines "in stock" once, so `len()` and looping can never disagree. `iter()` (NB22) turns the list into an iterator. Wrapping a dict (composition) instead of subclassing it keeps the rules impossible to bypass (see Q33).

</details>

In [ ]:
# your code here


### NB21-Q21 · L3 · Container protocol · Predict

`Squares` only defines `__getitem__`. Predict the output.

**What does this print?**

```python
class Squares:
    def __init__(self, count):
        self.count = count

    def __getitem__(self, index):
        if index >= self.count:
            raise IndexError(index)
        return index * index


sq = Squares(5)
print(list(sq))
print(9 in sq, 10 in sq)
print(sq[-1])
```

<details><summary><b>Hint</b></summary>

With no `__iter__`, Python loops by calling `__getitem__(0)`, `(1)`, `(2)`... until it gets `IndexError`. Does the method do anything special with `-1`?

</details>

<details><summary><b>Expected output</b></summary>

```text
[0, 1, 4, 9, 16]
True False
1
```

</details>

<details><summary><b>Solution</b></summary>

Python's old sequence protocol makes `list()` and `in` work from `__getitem__` alone: it asks for index 0, 1, 2... and stops at `IndexError`. `9 in sq` finds `9`; `10` is never produced. `sq[-1]` is not "the last one": your method gets `-1`, which passes the check, and returns `(-1) * (-1)`, which is `1`.

</details>

<details><summary><b>Senior dev solution</b></summary>

Negative indexes and slices are your job: built-in lists handle them, custom classes don't unless you write it. Seniors implement `__len__` and convert negative indexes with `index += len(self)` before checking.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB21-Q22 · L3 · __call__ · Write

Write a `Discount` class: `Discount(10)` means 10% off. Calling the object
with a price returns the discounted price rounded to 2 decimals. Print
whether the object is `callable()`, then the discounted list of `prices`:

```text
True
[18.0, 31.95, 8.99]
```

**Starting code (already in the cell below):**

```python
prices = [20, 35.5, 9.99]
```

<details><summary><b>Hint</b></summary>

`obj(price)` calls `obj.__call__(price)`.

</details>

<details><summary><b>Expected output</b></summary>

```text
True
[18.0, 31.95, 8.99]
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Discount:
    def __init__(self, percent):
        self.percent = percent

    def __call__(self, price):
        return round(price * (1 - self.percent / 100), 2)


ten_off = Discount(10)
print(callable(ten_off))
print([ten_off(p) for p in prices])
```

`__call__` makes an object usable anywhere a function is expected, like `map()` or a `key=`. Unlike a plain function, it carries its own settings (`percent`) as attributes you can inspect or change.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Discount:
    def __init__(self, percent):
        if not 0 <= percent <= 100:
            raise ValueError("percent must be between 0 and 100")
        self.percent = percent

    def __call__(self, price):
        return round(price * (1 - self.percent / 100), 2)

    def __repr__(self):
        return f"Discount({self.percent})"


ten_off = Discount(10)
print(callable(ten_off))
print(list(map(ten_off, prices)))
```

A callable object is a closure (NB09) with a name, a `repr` and room for validation. It's a good fit when the "function" has configuration you want to log or test.

</details>

In [ ]:
prices = [20, 35.5, 9.99]

# your code here


### NB21-Q23 · L3 · dataclasses · Write

Write a dataclass `Cart` with `owner` (str) and `items` (a list that
starts empty for **every** cart). Make two carts, add `"tea"` to Ana's,
and print both:

```text
Cart(owner='ana', items=['tea'])
Cart(owner='ben', items=[])
```

<details><summary><b>Hint</b></summary>

A list default can't be written as `= []` in a dataclass. `field(default_factory=...)` takes a function that builds a fresh default for each object.

</details>

<details><summary><b>Expected output</b></summary>

```text
Cart(owner='ana', items=['tea'])
Cart(owner='ben', items=[])
```

</details>

<details><summary><b>Solution</b></summary>

```python
from dataclasses import dataclass, field


@dataclass
class Cart:
    owner: str
    items: list = field(default_factory=list)


ana = Cart("ana")
ben = Cart("ben")
ana.items.append("tea")
print(ana)
print(ben)
```

`default_factory=list` calls `list()` once per new cart, so each cart gets its own empty list. This is the dataclass answer to the mutable default argument trap from NB07.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from dataclasses import dataclass, field


@dataclass
class Cart:
    owner: str
    items: list[str] = field(default_factory=list)

    def add(self, item):
        self.items.append(item)


ana = Cart("ana")
ben = Cart("ben")
ana.add("tea")
print(ana)
print(ben)
```

`list[str]` documents what the list holds (NB26). An `add()` method keeps callers from reaching into `items` directly, which leaves room to add rules (stock checks, limits) later.

</details>

In [ ]:
# your code here


### NB21-Q24 · L3 · dataclasses · Error

A list default written the obvious way. Which error does this raise?

**Which error does this raise?**

```python
from dataclasses import dataclass


@dataclass
class Order:
    customer: str
    lines: list = []


print("class created")
```

<details><summary><b>Hint</b></summary>

The problem shows up when the class is *created*, before any object exists. Recall the mutable default argument from NB07.

</details>

<details><summary><b>Expected output</b></summary>

```text
ValueError
```

</details>

<details><summary><b>Solution</b></summary>

`ValueError: mutable default <class 'list'> for field lines is not allowed: use default_factory`. A plain `= []` would be shared by every order, so `@dataclass` refuses it when the class is built. That's why nothing is printed.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from dataclasses import dataclass, field


@dataclass
class Order:
    customer: str
    lines: list = field(default_factory=list)


print("class created")
```

This check is one of the best reasons to prefer dataclasses: a hand-written `__init__(self, lines=[])` has the same bug and no warning at all.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB21-Q25 · L3 · Operator overloading · Write

`sum()` starts adding from `0`, so `sum(wallet)` first does `0 + Money(...)`.
Write a `Money` class (stores `cents`) with `__add__`, a `__radd__` that
makes `sum()` work, and a `__repr__` like `Money(1250)`. Print
`sum(wallet)`.

<details><summary><b>Hint</b></summary>

`0 + money` first tries `int.__add__`, which gives up, then calls `money.__radd__(0)`. Treat `0` as "nothing to add".

</details>

<details><summary><b>Expected output</b></summary>

```text
Money(1250)
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Money:
    def __init__(self, cents):
        self.cents = cents

    def __add__(self, other):
        return Money(self.cents + other.cents)

    def __radd__(self, other):
        if other == 0:
            return self
        return self + other

    def __repr__(self):
        return f"Money({self.cents})"


wallet = [Money(500), Money(250), Money(500)]
print(sum(wallet))
```

The `r` in `__radd__` means *reflected*: Python calls it when the object is on the **right** of `+` and the left side couldn't handle it. Only the first step of `sum()` hits `0 + Money(500)`. After that it's `Money + Money`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Money:
    def __init__(self, cents):
        self.cents = cents

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return Money(self.cents + other.cents)

    def __radd__(self, other):
        if other == 0:
            return self
        return NotImplemented

    def __repr__(self):
        return f"Money({self.cents})"


wallet = [Money(500), Money(250), Money(500)]
print(sum(wallet))
```

The senior `__radd__` accepts exactly `0` and refuses everything else, so `5 + Money(1)` is a clear `TypeError` rather than an `AttributeError` about `.cents`. Another option is `sum(wallet, Money(0))`, which never adds an int at all.

</details>

In [ ]:
# your code here


### NB21-Q26 · L3 · dataclasses · Refactor

This class writes `__init__`, `__repr__` and `__eq__` by hand. Replace it
with a dataclass. The output must stay the same.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
class Employee:
    def __init__(self, name, dept, salary):
        self.name = name
        self.dept = dept
        self.salary = salary

    def __repr__(self):
        return f"Employee(name={self.name!r}, dept={self.dept!r}, salary={self.salary!r})"

    def __eq__(self, other):
        if not isinstance(other, Employee):
            return NotImplemented
        return (self.name, self.dept, self.salary) == (other.name, other.dept, other.salary)


a = Employee("Ana", "IT", 52000)
b = Employee("Ana", "IT", 52000)
print(a)
print(a == b)
```

<details><summary><b>Hint</b></summary>

`@dataclass` writes exactly these three methods from the field list.

</details>

<details><summary><b>Expected output</b></summary>

```text
Employee(name='Ana', dept='IT', salary=52000)
True
```

</details>

<details><summary><b>Solution</b></summary>

```python
from dataclasses import dataclass


@dataclass
class Employee:
    name: str
    dept: str
    salary: int


a = Employee("Ana", "IT", 52000)
b = Employee("Ana", "IT", 52000)
print(a)
print(a == b)
```

The dataclass `repr` uses the same `field=value!r` format, and its `__eq__` compares the fields in order as a tuple, exactly like the hand-written one. Fifteen lines become four, and adding a field now updates all three methods at once.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from dataclasses import dataclass


@dataclass(slots=True)
class Employee:
    name: str
    dept: str
    salary: int


a = Employee("Ana", "IT", 52000)
b = Employee("Ana", "IT", 52000)
print(a)
print(a == b)
```

Hand-written boilerplate drifts: someone adds a field and forgets `__eq__`. Seniors default to `@dataclass` for classes that mainly hold data, and add `slots=True` (Q27) when there will be many objects.

</details>

In [ ]:
class Employee:
    def __init__(self, name, dept, salary):
        self.name = name
        self.dept = dept
        self.salary = salary

    def __repr__(self):
        return f"Employee(name={self.name!r}, dept={self.dept!r}, salary={self.salary!r})"

    def __eq__(self, other):
        if not isinstance(other, Employee):
            return NotImplemented
        return (self.name, self.dept, self.salary) == (other.name, other.dept, other.salary)


a = Employee("Ana", "IT", 52000)
b = Employee("Ana", "IT", 52000)
print(a)
print(a == b)


---

## L4 · Somewhat Difficult

### NB21-Q27 · L4 · __slots__ · Write

Write a `SensorReading` class with `__slots__` for `sensor` and `value`.
Create `SensorReading("t1", 21.5)`, print its two fields, then try to set
a new attribute `unit`. Catch the error and print `cannot add unit`.
Finally print whether the object has a `__dict__`:

```text
t1 21.5
cannot add unit
False
```

<details><summary><b>Hint</b></summary>

`__slots__ = ("sensor", "value")` goes in the class body. Adding an attribute that isn't listed raises `AttributeError`. `hasattr()` checks for `__dict__`.

</details>

<details><summary><b>Expected output</b></summary>

```text
t1 21.5
cannot add unit
False
```

</details>

<details><summary><b>Solution</b></summary>

```python
class SensorReading:
    __slots__ = ("sensor", "value")

    def __init__(self, sensor, value):
        self.sensor = sensor
        self.value = value


reading = SensorReading("t1", 21.5)
print(reading.sensor, reading.value)
try:
    reading.unit = "C"
except AttributeError:
    print("cannot add unit")
print(hasattr(reading, "__dict__"))
```

Normally each object keeps its attributes in a `__dict__`, which can grow. `__slots__` replaces that dict with fixed slots, so objects are smaller and unknown attributes are refused. Useful when you create millions of small objects.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from dataclasses import dataclass


@dataclass(slots=True)
class SensorReading:
    sensor: str
    value: float


reading = SensorReading("t1", 21.5)
print(reading.sensor, reading.value)
try:
    reading.unit = "C"
except AttributeError:
    print("cannot add unit")
print(hasattr(reading, "__dict__"))
```

`@dataclass(slots=True)` writes `__slots__` from the fields, so the list can't fall out of step with `__init__`. The trade-off: slotted objects can't get new attributes on the fly, which is usually exactly what you want.

</details>

In [ ]:
# your code here


### NB21-Q28 · L4 · dataclasses · Write

Write a frozen, ordered dataclass `Money` with fields `currency` (str)
and `cents` (int):

- `__post_init__` raises `ValueError(f"unknown currency {currency}")`
  unless the currency is `"USD"` or `"EUR"`
- `+` adds two `Money` values, raising
  `ValueError(f"cannot add {a} and {b}")` (the two currency codes) if they differ

Then run this script:

```py
prices = [Money("USD", 1999), Money("EUR", 500), Money("USD", 250)]
print(sorted(prices))
print(Money("USD", 250) + Money("USD", 1999))
try:
    Money("USD", 1) + Money("EUR", 1)
except ValueError as e:
    print("error:", e)
try:
    prices[0].cents = 0
except FrozenInstanceError:
    print("frozen")
print(len({Money("USD", 250), Money("USD", 250)}))
try:
    Money("GBP", 5)
except ValueError as e:
    print("error:", e)
```

<details><summary><b>Hint</b></summary>

`@dataclass(frozen=True, order=True)`. `order=True` compares the fields in the order you list them. `FrozenInstanceError` lives in `dataclasses`.

</details>

<details><summary><b>Expected output</b></summary>

```text
[Money(currency='EUR', cents=500), Money(currency='USD', cents=250), Money(currency='USD', cents=1999)]
Money(currency='USD', cents=2249)
error: cannot add USD and EUR
frozen
1
error: unknown currency GBP
```

</details>

<details><summary><b>Solution</b></summary>

```python
from dataclasses import dataclass, FrozenInstanceError

CURRENCIES = ("USD", "EUR")


@dataclass(frozen=True, order=True)
class Money:
    currency: str
    cents: int

    def __post_init__(self):
        if self.currency not in CURRENCIES:
            raise ValueError(f"unknown currency {self.currency}")

    def __add__(self, other):
        if self.currency != other.currency:
            raise ValueError(f"cannot add {self.currency} and {other.currency}")
        return Money(self.currency, self.cents + other.cents)


prices = [Money("USD", 1999), Money("EUR", 500), Money("USD", 250)]
print(sorted(prices))
print(Money("USD", 250) + Money("USD", 1999))
try:
    Money("USD", 1) + Money("EUR", 1)
except ValueError as e:
    print("error:", e)
try:
    prices[0].cents = 0
except FrozenInstanceError:
    print("frozen")
print(len({Money("USD", 250), Money("USD", 250)}))
try:
    Money("GBP", 5)
except ValueError as e:
    print("error:", e)
```

`__post_init__` runs right after the generated `__init__`, which is the place for validation. `order=True` sorts by `(currency, cents)`, so currencies are grouped first. `frozen=True` blocks changes and, together with `__eq__`, makes the objects hashable, so the set holds `1` item.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from dataclasses import dataclass, replace, FrozenInstanceError

CURRENCIES = frozenset({"USD", "EUR"})


@dataclass(frozen=True, order=True)
class Money:
    currency: str
    cents: int

    def __post_init__(self):
        if self.currency not in CURRENCIES:
            raise ValueError(f"unknown currency {self.currency}")

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        if self.currency != other.currency:
            raise ValueError(f"cannot add {self.currency} and {other.currency}")
        return replace(self, cents=self.cents + other.cents)


prices = [Money("USD", 1999), Money("EUR", 500), Money("USD", 250)]
print(sorted(prices))
print(Money("USD", 250) + Money("USD", 1999))
try:
    Money("USD", 1) + Money("EUR", 1)
except ValueError as e:
    print("error:", e)
try:
    prices[0].cents = 0
except FrozenInstanceError:
    print("frozen")
print(len({Money("USD", 250), Money("USD", 250)}))
try:
    Money("GBP", 5)
except ValueError as e:
    print("error:", e)
```

Field order is a design decision when `order=True`: listing `currency` first stops `$5` from being sorted between two euro amounts. `dataclasses.replace` builds a changed copy and runs `__post_init__` again. The mini-project takes this further with `Decimal` (NB16).

</details>

In [ ]:
# your code here


### NB21-Q29 · L4 · Inheritance and MRO · Predict

A diamond: `Child` inherits from `Left` and `Right`, which both inherit from `Base`. Predict the output.

**What does this print?**

```python
class Base:
    def describe(self):
        return ["Base"]


class Left(Base):
    def describe(self):
        return ["Left"] + super().describe()


class Right(Base):
    def describe(self):
        return ["Right"] + super().describe()


class Child(Left, Right):
    def describe(self):
        return ["Child"] + super().describe()


print([cls.__name__ for cls in Child.__mro__])
print(Child().describe())
print(Left().describe())
```

<details><summary><b>Hint</b></summary>

`super()` doesn't mean "my parent". It means "the next class in the MRO of the object I'm working on".

</details>

<details><summary><b>Expected output</b></summary>

```text
['Child', 'Left', 'Right', 'Base', 'object']
['Child', 'Left', 'Right', 'Base']
['Left', 'Base']
```

</details>

<details><summary><b>Solution</b></summary>

The MRO (method resolution order) lists each class once, children before parents, left before right: `Child, Left, Right, Base, object`. For a `Child` object, `super()` inside `Left` moves to `Right`, **not** `Base`. For a plain `Left` object, the MRO is `Left, Base, object`, so there `super()` goes straight to `Base`.

</details>

<details><summary><b>Senior dev solution</b></summary>

This is why it's called *cooperative* `super()`: every class calls `super()` once, and the MRO makes sure each class runs exactly once. `Child.mro()` and `help(Child)` both print the order when you're debugging.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB21-Q30 · L4 · Inheritance and MRO · Fix

`Base setup` runs twice. Fix the classes so every setup message prints
exactly once, in MRO order:

```text
Base setup
Cached setup
Logged setup
Service setup
```

**Buggy code (copied into the cell below):**

```python
class Base:
    def __init__(self):
        print("Base setup")


class Logged(Base):
    def __init__(self):
        Base.__init__(self)
        print("Logged setup")


class Cached(Base):
    def __init__(self):
        Base.__init__(self)
        print("Cached setup")


class Service(Logged, Cached):
    def __init__(self):
        Logged.__init__(self)
        Cached.__init__(self)
        print("Service setup")


Service()
```

<details><summary><b>Hint</b></summary>

Calling a parent by name always calls that exact class. `super().__init__()` follows the MRO instead: `Service, Logged, Cached, Base`.

</details>

<details><summary><b>Expected output</b></summary>

```text
Base setup
Cached setup
Logged setup
Service setup
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Base:
    def __init__(self):
        print("Base setup")


class Logged(Base):
    def __init__(self):
        super().__init__()
        print("Logged setup")


class Cached(Base):
    def __init__(self):
        super().__init__()
        print("Cached setup")


class Service(Logged, Cached):
    def __init__(self):
        super().__init__()
        print("Service setup")


Service()
```

With `super()`, `Service` calls `Logged`, whose `super()` calls `Cached` (the next class in the MRO, not `Base`), whose `super()` calls `Base`. Each class prints after its `super()` call returns, so the messages appear from `Base` back up to `Service`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Base:
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        print("Base setup")


class Logged(Base):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        print("Logged setup")


class Cached(Base):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        print("Cached setup")


class Service(Logged, Cached):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        print("Service setup")


Service()
```

In real cooperative hierarchies each class takes `**kwargs`, uses the arguments it knows, and passes the rest along, because no class knows who comes next in the MRO. Mixing `super()` with direct `Parent.__init__` calls is what breaks the chain.

</details>

In [ ]:
class Base:
    def __init__(self):
        print("Base setup")


class Logged(Base):
    def __init__(self):
        Base.__init__(self)
        print("Logged setup")


class Cached(Base):
    def __init__(self):
        Base.__init__(self)
        print("Cached setup")


class Service(Logged, Cached):
    def __init__(self):
        Logged.__init__(self)
        Cached.__init__(self)
        print("Service setup")


Service()


### NB21-Q31 · L4 · Abstract base classes · Write

Write an abstract class `Shape` with an abstract method `area()` and a
normal method `describe()` returning text like `Square with area 4.00`
(the class name plus the area to 2 decimals). Make `Square(side)` and
`Circle(radius)`. Print `describe()` for `Square(2)` and `Circle(1)`,
then try `Shape()` and print `Shape is abstract` when it fails.

<details><summary><b>Hint</b></summary>

`from abc import ABC, abstractmethod`. Inherit from `ABC` and put `@abstractmethod` on `area`. Creating a class that still has abstract methods raises `TypeError`. `type(self).__name__` gives the class name.

</details>

<details><summary><b>Expected output</b></summary>

```text
Square with area 4.00
Circle with area 3.14
Shape is abstract
```

</details>

<details><summary><b>Solution</b></summary>

```python
import math
from abc import ABC, abstractmethod


class Shape(ABC):
    @abstractmethod
    def area(self):
        pass

    def describe(self):
        return f"{type(self).__name__} with area {self.area():.2f}"


class Square(Shape):
    def __init__(self, side):
        self.side = side

    def area(self):
        return self.side ** 2


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2


for shape in [Square(2), Circle(1)]:
    print(shape.describe())
try:
    Shape()
except TypeError:
    print("Shape is abstract")
```

An abstract base class is a contract: every subclass *must* provide `area()`. `describe()` is written once in the base and works for every shape because it calls `self.area()`. Python refuses to create a `Shape` itself, since it has no real area.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import math
from abc import ABC, abstractmethod


class Shape(ABC):
    @abstractmethod
    def area(self):
        """Return the area in square units."""

    def describe(self):
        return f"{type(self).__name__} with area {self.area():.2f}"


class Square(Shape):
    def __init__(self, side):
        self.side = side

    def area(self):
        return self.side ** 2


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2


for shape in (Square(2), Circle(1)):
    print(shape.describe())
try:
    Shape()
except TypeError:
    print("Shape is abstract")
```

A docstring is a valid function body, so seniors document the contract instead of writing `pass`. This pattern (a base class with one abstract step and shared logic around it) is called the *template method*.

</details>

In [ ]:
# your code here


### NB21-Q32 · L4 · Abstract base classes · Error

The subclass named its method slightly wrong. Which error does this raise?

**Which error does this raise?**

```python
from abc import ABC, abstractmethod


class Exporter(ABC):
    @abstractmethod
    def export(self, rows):
        pass


class CsvExporter(Exporter):
    def export_rows(self, rows):
        return ",".join(rows)


print("classes defined")
exporter = CsvExporter()
print(exporter.export_rows(["a", "b"]))
```

<details><summary><b>Hint</b></summary>

Defining the classes works. The check happens when you try to create an object.

</details>

<details><summary><b>Expected output</b></summary>

```text
classes defined
TypeError
```

</details>

<details><summary><b>Solution</b></summary>

`TypeError: Can't instantiate abstract class CsvExporter...`. `export` is still abstract because `export_rows` is a different name. The ABC catches the typo the moment you create an object, not later when someone calls `export()`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from abc import ABC, abstractmethod


class Exporter(ABC):
    @abstractmethod
    def export(self, rows):
        pass


class CsvExporter(Exporter):
    def export(self, rows):
        return ",".join(rows)


print(CsvExporter().export(["a", "b"]))
```

This early failure is the point of `ABC`. With a plain base class whose `export` raises `NotImplementedError`, the typo would only surface when `export()` was finally called, maybe in production.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB21-Q33 · L4 · Subclassing built-ins · Predict

`LowerDict` lowercases keys in `__setitem__`. Predict the output.

**What does this print?**

```python
class LowerDict(dict):
    def __setitem__(self, key, value):
        super().__setitem__(key.lower(), value)


stock = LowerDict(Apple=3)
stock["Banana"] = 5
stock.update(Cherry=7)
print(sorted(stock))
```

<details><summary><b>Hint</b></summary>

Only one of the three ways of adding a key goes through your `__setitem__`. The built-in `dict` methods are written in C and don't call it.

</details>

<details><summary><b>Expected output</b></summary>

```text
['Apple', 'Cherry', 'banana']
```

</details>

<details><summary><b>Solution</b></summary>

Only `stock["Banana"] = 5` calls your `__setitem__`. The `dict` constructor and `update()` are C code that store keys directly, skipping your override. So `Apple` and `Cherry` keep their capitals: `['Apple', 'Cherry', 'banana']`.

</details>

<details><summary><b>Senior dev solution</b></summary>

This is the classic trap with subclassing `dict`, `list` and `str`: overriding one method doesn't change the others. Q34 shows the fix.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB21-Q34 · L4 · Subclassing built-ins · Fix

The same `LowerDict`. Fix it so **every** way of adding a key lowercases
it, and the script prints `['apple', 'banana', 'cherry']`.

**Buggy code (copied into the cell below):**

```python
class LowerDict(dict):
    def __setitem__(self, key, value):
        super().__setitem__(key.lower(), value)


stock = LowerDict(Apple=3)
stock["Banana"] = 5
stock.update(Cherry=7)
print(sorted(stock))
```

<details><summary><b>Hint</b></summary>

`collections.UserDict` is a dict written in Python whose other methods all go through `__setitem__`.

</details>

<details><summary><b>Expected output</b></summary>

```text
['apple', 'banana', 'cherry']
```

</details>

<details><summary><b>Solution</b></summary>

```python
from collections import UserDict


class LowerDict(UserDict):
    def __setitem__(self, key, value):
        super().__setitem__(key.lower(), value)


stock = LowerDict(Apple=3)
stock["Banana"] = 5
stock.update(Cherry=7)
print(sorted(stock))
```

`UserDict` exists for exactly this. It stores the real data in `self.data`, and its constructor and `update()` are built on `self[key] = value`, so your `__setitem__` sees every key.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from collections import UserDict


class LowerDict(UserDict):
    def __setitem__(self, key, value):
        super().__setitem__(key.lower(), value)

    def __getitem__(self, key):
        return super().__getitem__(key.lower())

    def __contains__(self, key):
        return key.lower() in self.data


stock = LowerDict(Apple=3)
stock["Banana"] = 5
stock.update(Cherry=7)
print(sorted(stock))
```

A case-insensitive dict must also lowercase on the way *in* to lookups, or `stock["APPLE"]` fails. Seniors list every way a value can enter or be found, and cover them all. `UserList` and `UserString` exist for the same reason.

</details>

In [ ]:
class LowerDict(dict):
    def __setitem__(self, key, value):
        super().__setitem__(key.lower(), value)


stock = LowerDict(Apple=3)
stock["Banana"] = 5
stock.update(Cherry=7)
print(sorted(stock))


### NB21-Q35 · L4 · dataclasses · Write [3.13+]

`Settings` is frozen, so you can't change it. Use `copy.replace()` to make
a changed copy. Print `base`, a copy with `debug=True`, and whether the
two are equal. Then try a copy with `port=70000`: `__post_init__` should
reject it, and you print `error: port out of range`.

**Starting code (already in the cell below):**

```python
import copy
from dataclasses import dataclass


@dataclass(frozen=True)
class Settings:
    host: str
    port: int
    debug: bool = False

    def __post_init__(self):
        if not 1 <= self.port <= 65535:
            raise ValueError("port out of range")


base = Settings("localhost", 8000)
```

<details><summary><b>Hint</b></summary>

`copy.replace(obj, field=new_value)` builds a new object through `__init__`, so validation runs again.

</details>

<details><summary><b>Expected output</b></summary>

```text
Settings(host='localhost', port=8000, debug=False)
Settings(host='localhost', port=8000, debug=True)
False
error: port out of range
```

</details>

<details><summary><b>Solution</b></summary>

```python
dev = copy.replace(base, debug=True)
print(base)
print(dev)
print(base == dev)
try:
    copy.replace(base, port=70000)
except ValueError as e:
    print("error:", e)
```

Frozen objects are never edited; you make a new one with some fields changed. `copy.replace()` (new in 3.13) does that for dataclasses, named tuples and dates. It calls `__init__`, so `__post_init__` checks the new port.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from dataclasses import replace

dev = replace(base, debug=True)
print(base)
print(dev)
print(base == dev)
try:
    replace(base, port=70000)
except ValueError as e:
    print("error:", e)
```

`dataclasses.replace` has done the same job since Python 3.7, so seniors use it in code that must run on older versions. `copy.replace` is the general version that also works on `namedtuple` and `datetime` objects.

</details>

In [ ]:
import copy
from dataclasses import dataclass


@dataclass(frozen=True)
class Settings:
    host: str
    port: int
    debug: bool = False

    def __post_init__(self):
        if not 1 <= self.port <= 65535:
            raise ValueError("port out of range")


base = Settings("localhost", 8000)

# your code here


### NB21-Q36 · L4 · Properties · Refactor

`get_target()` and `set_target()` are Java style. Refactor them into a
`target` property with the same validation. Keep the output the same.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
class Thermostat:
    def __init__(self, target):
        self._target = None
        self.set_target(target)

    def get_target(self):
        return self._target

    def set_target(self, value):
        if not 5 <= value <= 30:
            raise ValueError("target must be between 5 and 30")
        self._target = value


t = Thermostat(21)
t.set_target(t.get_target() + 2)
print(t.get_target())
try:
    t.set_target(40)
except ValueError as e:
    print("rejected:", e)
print(t.get_target())
```

<details><summary><b>Hint</b></summary>

`@property` for the getter, `@target.setter` for the setter. Then `t.target += 2` reads and writes through both.

</details>

<details><summary><b>Expected output</b></summary>

```text
23
rejected: target must be between 5 and 30
23
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Thermostat:
    def __init__(self, target):
        self.target = target

    @property
    def target(self):
        return self._target

    @target.setter
    def target(self, value):
        if not 5 <= value <= 30:
            raise ValueError("target must be between 5 and 30")
        self._target = value


t = Thermostat(21)
t.target += 2
print(t.target)
try:
    t.target = 40
except ValueError as e:
    print("rejected:", e)
print(t.target)
```

`t.target += 2` calls the getter, adds 2, then calls the setter, so the validation still runs. The `self._target = None` line is no longer needed because `__init__` assigns through the property.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
MIN_C, MAX_C = 5, 30


class Thermostat:
    def __init__(self, target):
        self.target = target

    @property
    def target(self):
        return self._target

    @target.setter
    def target(self, value):
        if not MIN_C <= value <= MAX_C:
            raise ValueError(f"target must be between {MIN_C} and {MAX_C}")
        self._target = value


t = Thermostat(21)
t.target += 2
print(t.target)
try:
    t.target = 40
except ValueError as e:
    print("rejected:", e)
print(t.target)
```

Named limits keep the check and its error message in step: change `MAX_C` and both update. Callers read `t.target` like any attribute, which is the Python way: start simple, add a property only when you need control.

</details>

In [ ]:
class Thermostat:
    def __init__(self, target):
        self._target = None
        self.set_target(target)

    def get_target(self):
        return self._target

    def set_target(self, value):
        if not 5 <= value <= 30:
            raise ValueError("target must be between 5 and 30")
        self._target = value


t = Thermostat(21)
t.set_target(t.get_target() + 2)
print(t.get_target())
try:
    t.set_target(40)
except ValueError as e:
    print("rejected:", e)
print(t.get_target())


### NB21-Q37 · L4 · Operator overloading · Fix

`Tag("sale") == "sale"` should be `False` (and `!=` should be `True`), but
the output looks wrong. Fix it so the script prints:

```text
True
False
True
```

**Buggy code (copied into the cell below):**

```python
class Tag:
    def __init__(self, name):
        self.name = name

    def __eq__(self, other):
        if not isinstance(other, Tag):
            return NotImplementedError
        return self.name == other.name


print(Tag("sale") == Tag("sale"))
print(Tag("sale") == "sale")
print(Tag("sale") != "sale")
```

<details><summary><b>Hint</b></summary>

Look closely at the name being returned. One is a special value that means "I can't compare these". The other is an exception class.

</details>

<details><summary><b>Expected output</b></summary>

```text
True
False
True
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Tag:
    def __init__(self, name):
        self.name = name

    def __eq__(self, other):
        if not isinstance(other, Tag):
            return NotImplemented
        return self.name == other.name


print(Tag("sale") == Tag("sale"))
print(Tag("sale") == "sale")
print(Tag("sale") != "sale")
```

`NotImplementedError` is an exception *class*. Returning it hands back a truthy object, so `==` printed the class itself and `!=` (which negates `__eq__`) gave `False`. `NotImplemented` is the special value that tells Python to try the other side, then fall back to identity: `False`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from dataclasses import dataclass


@dataclass(frozen=True)
class Tag:
    name: str


print(Tag("sale") == Tag("sale"))
print(Tag("sale") == "sale")
print(Tag("sale") != "sale")
```

Rule of thumb: you *return* `NotImplemented` from operator methods, and you *raise* `NotImplementedError` from methods a subclass must override. A dataclass writes the correct `__eq__` and never mixes them up.

</details>

In [ ]:
class Tag:
    def __init__(self, name):
        self.name = name

    def __eq__(self, other):
        if not isinstance(other, Tag):
            return NotImplementedError
        return self.name == other.name


print(Tag("sale") == Tag("sale"))
print(Tag("sale") == "sale")
print(Tag("sale") != "sale")


---

## L5 · Hard

### NB21-Q38 · L5 · Mixins · Write

Write two mixins and use them on `User` and its subclass `Admin`:

- `ReprMixin.__repr__` builds `ClassName(field=value, ...)` from the
  object's attributes, in the order they were set, skipping names that
  start with `_`
- `JsonMixin.to_json()` returns `json.dumps(...)` of the same public
  attributes with `sort_keys=True`

`User(name, email)` also stores `self._password_hash = "x9f"`, which must
never appear. `Admin(name, email, level)` adds `level`. Print:

```text
User(name='ana', email='ana@example.com')
{"email": "ana@example.com", "name": "ana"}
Admin(name='raj', email='raj@example.com', level=2)
{"email": "raj@example.com", "level": 2, "name": "raj"}
```

<details><summary><b>Hint</b></summary>

`vars(self)` is the object's attribute dict, in insertion order. Build a filtered dict once in a helper both mixins can use, and use `type(self).__name__` so `Admin` prints its own name.

</details>

<details><summary><b>Expected output</b></summary>

```text
User(name='ana', email='ana@example.com')
{"email": "ana@example.com", "name": "ana"}
Admin(name='raj', email='raj@example.com', level=2)
{"email": "raj@example.com", "level": 2, "name": "raj"}
```

</details>

<details><summary><b>Solution</b></summary>

```python
import json


class PublicFieldsMixin:
    def public_fields(self):
        return {key: value for key, value in vars(self).items() if not key.startswith("_")}


class ReprMixin(PublicFieldsMixin):
    def __repr__(self):
        parts = [f"{key}={value!r}" for key, value in self.public_fields().items()]
        return f"{type(self).__name__}({', '.join(parts)})"


class JsonMixin(PublicFieldsMixin):
    def to_json(self):
        return json.dumps(self.public_fields(), sort_keys=True)


class User(ReprMixin, JsonMixin):
    def __init__(self, name, email):
        self.name = name
        self.email = email
        self._password_hash = "x9f"


class Admin(User):
    def __init__(self, name, email, level):
        super().__init__(name, email)
        self.level = level


for person in [User("ana", "ana@example.com"), Admin("raj", "raj@example.com", 2)]:
    print(person)
    print(person.to_json())
```

A mixin is a small class that adds one ability and isn't meant to be used alone. Both mixins share `public_fields()`, so the "skip private attributes" rule exists once. `Admin` sets `level` after `super().__init__()`, which is why `level` comes last in its repr, and `type(self).__name__` makes the same `__repr__` say `Admin`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import json


class PublicFieldsMixin:
    def public_fields(self):
        return {key: value for key, value in vars(self).items() if not key.startswith("_")}


class ReprMixin(PublicFieldsMixin):
    def __repr__(self):
        parts = ", ".join(f"{key}={value!r}" for key, value in self.public_fields().items())
        return f"{type(self).__name__}({parts})"


class JsonMixin(PublicFieldsMixin):
    def to_json(self):
        return json.dumps(self.public_fields(), sort_keys=True)


class User(ReprMixin, JsonMixin):
    def __init__(self, name, email):
        self.name = name
        self.email = email
        self._password_hash = "x9f"


class Admin(User):
    def __init__(self, name, email, level):
        super().__init__(name, email)
        self.level = level


for person in (User("ana", "ana@example.com"), Admin("raj", "raj@example.com", 2)):
    print(person)
    print(person.to_json())
```

Seniors keep mixins tiny, stateless (no `__init__`) and named `...Mixin`, and list them before the main base class. A shared private-field filter is also a security habit: a password hash that leaks into a repr ends up in log files.

</details>

In [ ]:
# your code here


### NB21-Q39 · L5 · Operator overloading · Write

Write an immutable `Matrix` class (store rows as a tuple of tuples) with:

- `__repr__` like `Matrix([[1, 2], [3, 4]])`
- `@` (`__matmul__`) for matrix multiplication, raising
  `ValueError("shape mismatch")` when the shapes don't fit
- `==` and `__hash__` based on the rows
- indexing: `m[r, c]` gives one number, `m[r]` gives a row as a tuple, and
  `m[start:stop]` gives a new `Matrix` of those rows

Run this script:

```py
a = Matrix([[1, 2], [3, 4]])
b = Matrix([[5, 6], [7, 8]])
identity = Matrix([[1, 0], [0, 1]])
print(a @ b)
print(a @ identity == a)
print(a[1, 0], a[0])
print(Matrix([[1, 2], [3, 4], [5, 6]])[1:])
print(len({a, Matrix([[1, 2], [3, 4]]), b}))
try:
    a @ Matrix([[1, 2, 3]])
except ValueError as e:
    print("error:", e)
```

<details><summary><b>Hint</b></summary>

`m[1, 0]` passes the tuple `(1, 0)` to `__getitem__`; `m[1:]` passes a `slice` object. Check with `isinstance`. For `@`, `zip(*other.rows)` gives the columns of `other`.

</details>

<details><summary><b>Expected output</b></summary>

```text
Matrix([[19, 22], [43, 50]])
True
3 (1, 2)
Matrix([[3, 4], [5, 6]])
2
error: shape mismatch
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Matrix:
    def __init__(self, rows):
        self.rows = tuple(tuple(row) for row in rows)

    def __repr__(self):
        return f"Matrix({[list(row) for row in self.rows]})"

    def __eq__(self, other):
        if not isinstance(other, Matrix):
            return NotImplemented
        return self.rows == other.rows

    def __hash__(self):
        return hash(self.rows)

    def __matmul__(self, other):
        if len(self.rows[0]) != len(other.rows):
            raise ValueError("shape mismatch")
        columns = list(zip(*other.rows))
        return Matrix([[sum(x * y for x, y in zip(row, col)) for col in columns]
                       for row in self.rows])

    def __getitem__(self, key):
        if isinstance(key, tuple):
            r, c = key
            return self.rows[r][c]
        if isinstance(key, slice):
            return Matrix(self.rows[key])
        return self.rows[key]


a = Matrix([[1, 2], [3, 4]])
b = Matrix([[5, 6], [7, 8]])
identity = Matrix([[1, 0], [0, 1]])
print(a @ b)
print(a @ identity == a)
print(a[1, 0], a[0])
print(Matrix([[1, 2], [3, 4], [5, 6]])[1:])
print(len({a, Matrix([[1, 2], [3, 4]]), b}))
try:
    a @ Matrix([[1, 2, 3]])
except ValueError as e:
    print("error:", e)
```

The trap is hashing: a hash must never change, so the rows are stored as tuples, not lists. Equal matrices hash the same, so the set holds `2`. In `__matmul__`, each result cell is the sum of a row times a column. `__getitem__` checks the key's type, because `m[r, c]`, `m[r]` and `m[a:b]` all arrive at the same method.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Matrix:
    __slots__ = ("rows",)

    def __init__(self, rows):
        self.rows = tuple(tuple(row) for row in rows)

    def __repr__(self):
        return f"Matrix({[list(row) for row in self.rows]})"

    def __eq__(self, other):
        if not isinstance(other, Matrix):
            return NotImplemented
        return self.rows == other.rows

    def __hash__(self):
        return hash(self.rows)

    @property
    def shape(self):
        return len(self.rows), len(self.rows[0])

    def __matmul__(self, other):
        if not isinstance(other, Matrix):
            return NotImplemented
        if self.shape[1] != other.shape[0]:
            raise ValueError("shape mismatch")
        columns = list(zip(*other.rows))
        return Matrix([[sum(x * y for x, y in zip(row, col, strict=True)) for col in columns]
                       for row in self.rows])

    def __getitem__(self, key):
        match key:
            case (int() as r, int() as c):
                return self.rows[r][c]
            case slice():
                return Matrix(self.rows[key])
            case int():
                return self.rows[key]
        raise TypeError(f"invalid index {key!r}")


a = Matrix([[1, 2], [3, 4]])
b = Matrix([[5, 6], [7, 8]])
identity = Matrix([[1, 0], [0, 1]])
print(a @ b)
print(a @ identity == a)
print(a[1, 0], a[0])
print(Matrix([[1, 2], [3, 4], [5, 6]])[1:])
print(len({a, Matrix([[1, 2], [3, 4]]), b}))
try:
    a @ Matrix([[1, 2, 3]])
except ValueError as e:
    print("error:", e)
```

A `shape` property names the idea the error is about. `match` with class patterns (NB18) reads like the three indexing rules, and a final `TypeError` rejects keys like `m["x"]`, as lists do. For real numeric work, NumPy does all of this in fast C code.

</details>

In [ ]:
# your code here


### NB21-Q40 · L5 · dataclasses · Write

Write a dataclass `Task` that is ordered, keyword-only and slotted, with
fields `priority` (int), `due` (ISO date text like `"2026-10-08"`),
`title` (str) and `tags` (list, empty by default). Sorting and `==` must
use **only** `priority` and `due`; `tags` must also stay out of the repr.

Print the sorted tasks as `priority due title`, then whether the first and
last tasks in `specs` are equal, then try `Task(1, "2026-10-01", "x")`
and print `keyword arguments only` when it fails:

```text
1 2026-10-07 Reply to client
1 2026-10-09 Fix login bug
2 2026-10-08 Write report
2 2026-10-08 Book travel
True
keyword arguments only
```

**Starting code (already in the cell below):**

```python
specs = [
    {"priority": 2, "due": "2026-10-08", "title": "Write report"},
    {"priority": 1, "due": "2026-10-09", "title": "Fix login bug"},
    {"priority": 1, "due": "2026-10-07", "title": "Reply to client"},
    {"priority": 2, "due": "2026-10-08", "title": "Book travel"},
]
```

<details><summary><b>Hint</b></summary>

`@dataclass(order=True, kw_only=True, slots=True)`. `field(compare=False)` keeps a field out of `==` and ordering; `repr=False` keeps it out of the repr. Build each task with `Task(**spec)`.

</details>

<details><summary><b>Expected output</b></summary>

```text
1 2026-10-07 Reply to client
1 2026-10-09 Fix login bug
2 2026-10-08 Write report
2 2026-10-08 Book travel
True
keyword arguments only
```

</details>

<details><summary><b>Solution</b></summary>

```python
from dataclasses import dataclass, field


@dataclass(order=True, kw_only=True, slots=True)
class Task:
    priority: int
    due: str
    title: str = field(compare=False)
    tags: list = field(default_factory=list, compare=False, repr=False)


tasks = [Task(**spec) for spec in specs]
for task in sorted(tasks):
    print(task.priority, task.due, task.title)
print(tasks[0] == tasks[-1])
try:
    Task(1, "2026-10-01", "x")
except TypeError:
    print("keyword arguments only")
```

`order=True` compares the *compared* fields in order: priority, then due. ISO dates sort correctly as text. The two priority-2 tasks tie, and `sorted()` is stable, so they keep their original order. The trap: because `title` isn't compared, two *different* tasks are `==`. `kw_only=True` makes positional calls a `TypeError`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from dataclasses import dataclass, field
from datetime import date


@dataclass(order=True, kw_only=True, slots=True)
class Task:
    priority: int
    due: date
    title: str = field(compare=False)
    tags: list[str] = field(default_factory=list, compare=False, repr=False)

    def __post_init__(self):
        if isinstance(self.due, str):
            self.due = date.fromisoformat(self.due)


tasks = [Task(**spec) for spec in specs]
for task in sorted(tasks):
    print(task.priority, task.due, task.title)
print(tasks[0] == tasks[-1])
try:
    Task(1, "2026-10-01", "x")
except TypeError:
    print("keyword arguments only")
```

Storing a real `date` (NB15) means a typo like `2026-13-01` fails at once instead of sorting quietly in the wrong place. Seniors are wary of `compare=False` on identifying fields: if equality should mean "same task", use `sorted(tasks, key=...)` for ordering and keep `==` strict.

</details>

In [ ]:
specs = [
    {"priority": 2, "due": "2026-10-08", "title": "Write report"},
    {"priority": 1, "due": "2026-10-09", "title": "Fix login bug"},
    {"priority": 1, "due": "2026-10-07", "title": "Reply to client"},
    {"priority": 2, "due": "2026-10-08", "title": "Book travel"},
]

# your code here


### NB21-Q41 · L5 · Operator overloading · Predict

`+=` with and without `__iadd__`, when two names share one object. Predict the output.

**What does this print?**

```python
class Tally:
    def __init__(self, n):
        self.n = n

    def __add__(self, other):
        return Tally(self.n + other)

    def __repr__(self):
        return f"Tally({self.n})"


class Bag:
    def __init__(self, n):
        self.n = n

    def __add__(self, other):
        return Bag(self.n + other)

    def __iadd__(self, other):
        self.n += other
        return self

    def __repr__(self):
        return f"Bag({self.n})"


a = b = Tally(1)
a += 5
print(a, b)
c = d = Bag(1)
c += 5
print(c, d)
x = y = [1]
x += [2]
print(x, y)
```

<details><summary><b>Hint</b></summary>

`a += 5` first tries `a.__iadd__(5)`. If there's no `__iadd__`, it falls back to `a = a + 5`, which builds a new object. Which version changes the object both names point to?

</details>

<details><summary><b>Expected output</b></summary>

```text
Tally(6) Tally(1)
Bag(6) Bag(6)
[1, 2] [1, 2]
```

</details>

<details><summary><b>Solution</b></summary>

`Tally` has no `__iadd__`, so `a += 5` becomes `a = a + 5`: `a` now names a **new** object and `b` still names the old one. `Bag.__iadd__` changes the object in place and returns it, so `c` and `d` both see `Bag(6)`. Lists define `__iadd__` too, which is why `x += [2]` also changes `y`.

</details>

<details><summary><b>Senior dev solution</b></summary>

Rule: immutable types (int, str, tuple, frozen dataclasses) shouldn't define `__iadd__`; mutable containers may. `__iadd__` must `return self`. If it returns nothing, `c` becomes `None`.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB21-Q42 · L5 · Operator overloading · Write

Write a `Polynomial` class from coefficients, **lowest power first**
(`Polynomial([1, 2, 3])` is `3x^2 + 2x + 1`). It needs:

- `__repr__`: terms from the highest power down, joined with ` + `,
  skipping zero coefficients; power 0 is `c`, power 1 is `cx`, higher is
  `cx^n`; the zero polynomial is `0`
- `__call__(x)`: evaluate at `x`
- `+` with another polynomial or an int, from either side (so `5 + q` and
  `sum()` work)
- `==` and `len()` (number of coefficients), with trailing zero
  coefficients always removed

Run this script:

```py
p = Polynomial([1, 2, 3])
q = Polynomial([0, 1])
print(p)
print(p(2))
print(p + q)
print(5 + q)
print(sum([p, q, q]))
cancelled = p + Polynomial([0, 0, -3])
print(cancelled, cancelled == Polynomial([1, 2]), len(cancelled))
```

<details><summary><b>Hint</b></summary>

Trim trailing zeros once in `__init__`, and every other method can rely on it. Turn an int into `Polynomial([n])` before adding. `itertools.zip_longest(..., fillvalue=0)` adds lists of different lengths.

</details>

<details><summary><b>Expected output</b></summary>

```text
3x^2 + 2x + 1
17
3x^2 + 3x + 1
1x + 5
3x^2 + 4x + 1
2x + 1 True 2
```

</details>

<details><summary><b>Solution</b></summary>

```python
from itertools import zip_longest


class Polynomial:
    def __init__(self, coeffs):
        coeffs = list(coeffs)
        while coeffs and coeffs[-1] == 0:
            coeffs.pop()
        self.coeffs = coeffs

    def __repr__(self):
        terms = []
        for power in range(len(self.coeffs) - 1, -1, -1):
            c = self.coeffs[power]
            if c == 0:
                continue
            if power == 0:
                terms.append(f"{c}")
            elif power == 1:
                terms.append(f"{c}x")
            else:
                terms.append(f"{c}x^{power}")
        if not terms:
            return "0"
        return " + ".join(terms)

    def __call__(self, x):
        return sum(c * x ** power for power, c in enumerate(self.coeffs))

    def __add__(self, other):
        if isinstance(other, int):
            other = Polynomial([other])
        pairs = zip_longest(self.coeffs, other.coeffs, fillvalue=0)
        return Polynomial([a + b for a, b in pairs])

    def __radd__(self, other):
        return self + other

    def __eq__(self, other):
        return self.coeffs == other.coeffs

    def __len__(self):
        return len(self.coeffs)


p = Polynomial([1, 2, 3])
q = Polynomial([0, 1])
print(p)
print(p(2))
print(p + q)
print(5 + q)
print(sum([p, q, q]))
cancelled = p + Polynomial([0, 0, -3])
print(cancelled, cancelled == Polynomial([1, 2]), len(cancelled))
```

Trimming in `__init__` is the key design choice: `p + (-3x^2)` would otherwise keep a `0` on the end, and then `==` and `len()` give the wrong answer. `__radd__` handles `5 + q` and the `0 + p` that `sum()` starts with, by reusing `__add__` (addition is commutative).

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from itertools import zip_longest


class Polynomial:
    def __init__(self, coeffs):
        coeffs = list(coeffs)
        while coeffs and coeffs[-1] == 0:
            coeffs.pop()
        self.coeffs = tuple(coeffs)

    @staticmethod
    def _term(c, power):
        if power == 0:
            return f"{c}"
        return f"{c}x" if power == 1 else f"{c}x^{power}"

    def __repr__(self):
        terms = [self._term(c, power)
                 for power, c in reversed(list(enumerate(self.coeffs))) if c]
        return " + ".join(terms) or "0"

    def __call__(self, x):
        result = 0
        for c in reversed(self.coeffs):
            result = result * x + c
        return result

    def __add__(self, other):
        if isinstance(other, int):
            other = Polynomial([other])
        if not isinstance(other, Polynomial):
            return NotImplemented
        return Polynomial(a + b for a, b in zip_longest(self.coeffs, other.coeffs, fillvalue=0))

    __radd__ = __add__

    def __eq__(self, other):
        if not isinstance(other, Polynomial):
            return NotImplemented
        return self.coeffs == other.coeffs

    def __hash__(self):
        return hash(self.coeffs)

    def __len__(self):
        return len(self.coeffs)


p = Polynomial([1, 2, 3])
q = Polynomial([0, 1])
print(p)
print(p(2))
print(p + q)
print(5 + q)
print(sum([p, q, q]))
cancelled = p + Polynomial([0, 0, -3])
print(cancelled, cancelled == Polynomial([1, 2]), len(cancelled))
```

`__call__` uses Horner's method: one multiply and one add per coefficient, with no powers. Storing a tuple makes the polynomial immutable, which earns it a `__hash__`. `__radd__ = __add__` is safe only because addition is commutative; never alias `__rsub__` that way.

</details>

In [ ]:
# your code here


---

## Mini-project

Pull together everything from this notebook.

## NB21-P · L5 · Mini-project: Money Library · Write

Build a small money library and use it to price a shopping basket.

1. `Currency` is an `Enum` whose values are symbols: `USD = "$"`,
   `EUR = "€"`.
2. `CurrencyMismatch` is an exception (subclass `ValueError`).
3. `Money` is a frozen dataclass with `amount` (a `Decimal`) and
   `currency`. `__post_init__` converts the amount to `Decimal` (it may
   arrive as text or an int) and rounds it to cents with `ROUND_HALF_UP`.
   Because the class is frozen, store it with
   `object.__setattr__(self, "amount", ...)`.
4. `Money` supports `+` and `-` (same currency only, otherwise raise
   `CurrencyMismatch(f"cannot combine {a} and {b}")` with the currency
   **names**), `sum()` (via `__radd__`), `*` by a number from either side,
   `<` (and the rest via `@total_ordering`) within one currency, and
   `__str__` like `$12.50`.
5. `Money.split(n)` returns `n` amounts that add up exactly to the
   original: everyone gets the rounded-down share, and the leftover cents
   go one each to the first people.
6. `Converter(rates)` is callable: `convert(money, Currency.EUR)` uses
   `rates[(from_currency, to_currency)]`.

With the starting values, print exactly:

```text
Basket: $12.50 + $7.25 + $0.99 = $20.74
Most expensive: $12.50
Tax 8%: $1.66
Total: $22.40
Split 3 ways: $7.47, $7.47, $7.46
In EUR: €20.61
Error: cannot combine USD and EUR
```

**Starting code (already in the cell below):**

```python
from decimal import Decimal, ROUND_HALF_UP

prices = ["12.50", "7.25", "0.99"]
tax_rate = Decimal("0.08")
rates = {("USD", "EUR"): Decimal("0.92")}
```

<details><summary><b>Hint</b></summary>

Work in this order: `Currency`, `Money` with `__post_init__` and `__str__`, then the operators one at a time. For `split`, work in whole cents: `divmod(cents, n)` gives the base share and how many people get one extra cent. Key the rates by the currency **names** so the dict matches `rates`.

</details>

<details><summary><b>Expected output</b></summary>

```text
Basket: $12.50 + $7.25 + $0.99 = $20.74
Most expensive: $12.50
Tax 8%: $1.66
Total: $22.40
Split 3 ways: $7.47, $7.47, $7.46
In EUR: €20.61
Error: cannot combine USD and EUR
```

</details>

<details><summary><b>Solution</b></summary>

```python
from dataclasses import dataclass
from enum import Enum
from functools import total_ordering

CENT = Decimal("0.01")


class Currency(Enum):
    USD = "$"
    EUR = "€"


class CurrencyMismatch(ValueError):
    pass


@total_ordering
@dataclass(frozen=True)
class Money:
    amount: Decimal
    currency: Currency = Currency.USD

    def __post_init__(self):
        rounded = Decimal(self.amount).quantize(CENT, rounding=ROUND_HALF_UP)
        object.__setattr__(self, "amount", rounded)

    def _check(self, other):
        if self.currency is not other.currency:
            raise CurrencyMismatch(
                f"cannot combine {self.currency.name} and {other.currency.name}")

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        self._check(other)
        return Money(self.amount + other.amount, self.currency)

    def __radd__(self, other):
        if other == 0:
            return self
        return NotImplemented

    def __sub__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        self._check(other)
        return Money(self.amount - other.amount, self.currency)

    def __mul__(self, factor):
        if not isinstance(factor, (int, Decimal)):
            return NotImplemented
        return Money(self.amount * factor, self.currency)

    def __rmul__(self, factor):
        return self * factor

    def __lt__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        self._check(other)
        return self.amount < other.amount

    def __str__(self):
        return f"{self.currency.value}{self.amount}"

    def split(self, n):
        cents = int(self.amount * 100)
        share, extra = divmod(cents, n)
        parts = []
        for i in range(n):
            part_cents = share + 1 if i < extra else share
            parts.append(Money(Decimal(part_cents) / 100, self.currency))
        return parts


class Converter:
    def __init__(self, rates):
        self.rates = rates

    def __call__(self, money, to_currency):
        rate = self.rates[(money.currency.name, to_currency.name)]
        return Money(money.amount * rate, to_currency)


basket = [Money(p) for p in prices]
subtotal = sum(basket)
tax = subtotal * tax_rate
total = subtotal + tax
convert = Converter(rates)

print("Basket:", " + ".join(str(m) for m in basket), "=", subtotal)
print("Most expensive:", max(basket))
print("Tax 8%:", tax)
print("Total:", total)
print("Split 3 ways:", ", ".join(str(m) for m in total.split(3)))
print("In EUR:", convert(total, Currency.EUR))
try:
    total + Money("5", Currency.EUR)
except CurrencyMismatch as e:
    print("Error:", e)
```

Every amount goes through `__post_init__`, so money is always a `Decimal` with exactly two places, whether it started as text or a calculation. `@total_ordering` sits above `@dataclass` so it sees the finished class, and `max()` only needs `<`. `split` works in whole cents so nothing is lost: 2240 cents is 746 each with 2 cents left over. `Converter` is a callable object that carries its rate table.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from dataclasses import dataclass
from enum import Enum
from functools import total_ordering

CENT = Decimal("0.01")


class Currency(Enum):
    USD = "$"
    EUR = "€"


class CurrencyMismatch(ValueError):
    pass


@total_ordering
@dataclass(frozen=True, slots=True)
class Money:
    amount: Decimal
    currency: Currency = Currency.USD

    def __post_init__(self):
        rounded = Decimal(self.amount).quantize(CENT, rounding=ROUND_HALF_UP)
        object.__setattr__(self, "amount", rounded)

    def _same_currency(self, other):
        if not isinstance(other, Money):
            return False
        if self.currency is not other.currency:
            raise CurrencyMismatch(
                f"cannot combine {self.currency.name} and {other.currency.name}")
        return True

    def __add__(self, other):
        if not self._same_currency(other):
            return NotImplemented
        return Money(self.amount + other.amount, self.currency)

    def __radd__(self, other):
        return self if other == 0 else NotImplemented

    def __sub__(self, other):
        if not self._same_currency(other):
            return NotImplemented
        return Money(self.amount - other.amount, self.currency)

    def __mul__(self, factor):
        if isinstance(factor, bool) or not isinstance(factor, (int, Decimal)):
            return NotImplemented
        return Money(self.amount * factor, self.currency)

    __rmul__ = __mul__

    def __lt__(self, other):
        if not self._same_currency(other):
            return NotImplemented
        return self.amount < other.amount

    def __str__(self):
        return f"{self.currency.value}{self.amount}"

    def split(self, n):
        share, extra = divmod(int(self.amount / CENT), n)
        return [Money((share + (i < extra)) * CENT, self.currency) for i in range(n)]


class Converter:
    def __init__(self, rates):
        self.rates = dict(rates)

    def __call__(self, money, to_currency):
        if money.currency is to_currency:
            return money
        rate = self.rates[(money.currency.name, to_currency.name)]
        return Money(money.amount * rate, to_currency)


def joined(amounts, sep=", "):
    return sep.join(map(str, amounts))


basket = [Money(p) for p in prices]
subtotal = sum(basket)
tax = subtotal * tax_rate
total = subtotal + tax
convert = Converter(rates)

print(f"Basket: {joined(basket, ' + ')} = {subtotal}")
print(f"Most expensive: {max(basket)}")
print(f"Tax 8%: {tax}")
print(f"Total: {total}")
print(f"Split 3 ways: {joined(total.split(3))}")
print(f"In EUR: {convert(total, Currency.EUR)}")
try:
    total + Money("5", Currency.EUR)
except CurrencyMismatch as e:
    print(f"Error: {e}")
```

Both versions refuse `float` in `*`, because `Money * 1.1` would sneak a binary float into exact money (NB16); the senior also refuses `True`, which is an `int` (NB01). Converting to the same currency returns the money unchanged instead of needing a `(USD, USD)` rate. `share + (i < extra)` uses `True == 1`, a trick worth a comment in real code. Production libraries also store the currency's number of decimal places, because yen have none.

</details>

In [ ]:
from decimal import Decimal, ROUND_HALF_UP

prices = ["12.50", "7.25", "0.99"]
tax_rate = Decimal("0.08")
rates = {("USD", "EUR"): Decimal("0.92")}

# your code here


---

## Done

Next up: **NB22 · Iterators Generators** in `advanced/`.